In [ ]:
# The following models are in chronological order of testing and not in the order of their classification accuracies.
# Standard Parameters:
# BATCH_SIZE = 16
# EPOCHS = 20 
# LEARNING_RATE = 0.0001
# Image Size = 224x224
# If there is any change in these parameters, it will be explicitly mentioned in that particular cell.

In [66]:
# Necessary Imports
import os
import torch
from tqdm import tqdm
from torchvision import datasets
from torchvision.transforms import v2
from PIL import Image
torch.manual_seed(37)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [35]:
# Testing Function
def final_test(model,labels, preprocess, test_path):
    model.eval()
    correct = 0
    total = 0
    
    results_summary = {}

    print("Running validation on subset...")
    
    for class_folder in os.listdir(test_path):
        folder_path = os.path.join(test_path, class_folder)
        if not os.path.isdir(folder_path): continue
        
        results_summary[class_folder] = {"correct": 0, "total": 0}
        
        for img_name in tqdm(os.listdir(folder_path), desc=f"Testing {class_folder}"):

            img_path = os.path.join(folder_path, img_name)
            
            try:
                img = Image.open(img_path).convert('RGB')
            except:
                continue
                
            input_tensor = preprocess(img).unsqueeze(0).to(DEVICE)
            
            with torch.no_grad():
                output = model(input_tensor)
                _, pred_idx = torch.max(output, 1)
                
                predicted_name = labels[pred_idx.item()]
            
            if predicted_name.strip() == class_folder.strip():
                correct += 1
                results_summary[class_folder]["correct"] += 1
            
            total += 1
            results_summary[class_folder]["total"] += 1
    print(results_summary)

    print("\n" + "="*40)
    print(f"{'Class Name':<30} | {'Accuracy':<10}")
    print("-"*45)
    for cls, stats in results_summary.items():
        acc = (stats['correct']/stats['total'])*100 if stats['total'] > 0 else 0
        print(f"{cls:<30} | {acc:>8.2f}%")
    
    final_score = (correct / total) * 100
    print("="*40)
    print(f"OVERALL ACCURACY ON SUBSET: {final_score:.2f}%")

In [ ]:
# YOLO v11 + Efficient_Net_B4 (Pipeline)
# Trained on Plant_Doc + New_Plant_Diseases + Plant_Wild Dataset
# Tested on the dataset provided consisting 10 classes (excluding Tomato_Mosaic Virus, Tomato_Spider_Mites and Tomato_Target_Spot)

In [36]:
model_loaded = torch.load("Efficient_B4_Merged.pth",weights_only=False)

In [37]:
# Testing Block

full_dataset = datasets.ImageFolder("Merged_Dataset/cropped_training")

labels = full_dataset.classes

preprocess = v2.Compose([
    v2.ToTensor(),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Path to test images
TEST_DIR = "Merged_Dataset/cropped_testing" 

final_test(model_loaded,labels,preprocess, TEST_DIR)

c:\Anaconda\envs\env\Lib\site-packages\torchvision\transforms\v2\_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


Running validation on subset...


Testing Tomato_Tomato_Yellow_Leaf_Curl_Virus: 100%|██████████| 84/84 [00:02<00:00, 39.19it/s]

{'Potato_Early_blight': {'correct': 57, 'total': 87}, 'Potato_healthy': {'correct': 53, 'total': 63}, 'Potato_Lateblight': {'correct': 48, 'total': 104}, 'Tomato_Bacterial_spot': {'correct': 54, 'total': 100}, 'Tomato_Early_blight': {'correct': 31, 'total': 62}, 'Tomato_healthy': {'correct': 35, 'total': 71}, 'Tomato_Late_blight': {'correct': 50, 'total': 90}, 'Tomato_Leaf_mold': {'correct': 45, 'total': 73}, 'Tomato_Septoria_leaf_spot': {'correct': 49, 'total': 108}, 'Tomato_Tomato_Yellow_Leaf_Curl_Virus': {'correct': 64, 'total': 84}}

Class Name                     | Accuracy  
---------------------------------------------
Potato_Early_blight            |    65.52%
Potato_healthy                 |    84.13%
Potato_Lateblight              |    46.15%
Tomato_Bacterial_spot          |    54.00%
Tomato_Early_blight            |    50.00%
Tomato_healthy                 |    49.30%
Tomato_Late_blight             |    55.56%
Tomato_Leaf_mold               |    61.64%
Tomato_Septoria_leaf_s

In [ ]:
# YOLO v11 + Efficient_Net_B4 (Pipeline)
# Trained on Plant_Doc + New_Plant_Diseases + Plant_Wild Dataset, images were resized from 224x224 to 380x380
# Tested on the dataset provided consisting 10 classes (excluding Tomato_Mosaic Virus, Tomato_Spider_Mites and Tomato_Target_Spot)

In [38]:
model_loaded = torch.load("Efficient_B4_Merged_380.pth",weights_only=False)

In [39]:
# Testing Block

full_dataset = datasets.ImageFolder("Merged_Dataset/cropped_training")

labels = full_dataset.classes

preprocess = v2.Compose([
    v2.ToTensor(),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Path to test images
TEST_DIR = "Merged_Dataset/cropped_testing_380" 

final_test(model_loaded, labels,preprocess, TEST_DIR)

Running validation on subset...


Testing Tomato_Tomato_Yellow_Leaf_Curl_Virus: 100%|██████████| 84/84 [00:02<00:00, 39.56it/s]

{'Potato_Early_blight': {'correct': 52, 'total': 87}, 'Potato_healthy': {'correct': 46, 'total': 63}, 'Potato_Lateblight': {'correct': 45, 'total': 104}, 'Tomato_Bacterial_spot': {'correct': 59, 'total': 100}, 'Tomato_Early_blight': {'correct': 29, 'total': 62}, 'Tomato_healthy': {'correct': 29, 'total': 71}, 'Tomato_Late_blight': {'correct': 50, 'total': 90}, 'Tomato_Leaf_mold': {'correct': 57, 'total': 73}, 'Tomato_Septoria_leaf_spot': {'correct': 41, 'total': 108}, 'Tomato_Tomato_Yellow_Leaf_Curl_Virus': {'correct': 51, 'total': 84}}

Class Name                     | Accuracy  
---------------------------------------------
Potato_Early_blight            |    59.77%
Potato_healthy                 |    73.02%
Potato_Lateblight              |    43.27%
Tomato_Bacterial_spot          |    59.00%
Tomato_Early_blight            |    46.77%
Tomato_healthy                 |    40.85%
Tomato_Late_blight             |    55.56%
Tomato_Leaf_mold               |    78.08%
Tomato_Septoria_leaf_s

In [ ]:
# YOLO v11 + Efficient_Net_B4 (Pipeline)
# Trained on Plant_Doc + Plant_Wild Dataset for 40 epochs
# Tested on the dataset provided consisting 10 classes (excluding Tomato_Mosaic Virus, Tomato_Spider_Mites and Tomato_Target_Spot)

In [40]:
model_loaded = torch.load("Efficient_B4_Merged_Plant_Doc_Wild_Only.pth",weights_only=False)

In [41]:
# Testing Block

full_dataset = datasets.ImageFolder("Merged_Dataset(Plant_Doc+Plant_Wild)/cropped_training")

labels = full_dataset.classes

preprocess = v2.Compose([
    v2.ToTensor(),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Path to test images
TEST_DIR = "Merged_Dataset(Plant_Doc+Plant_Wild)/cropped_testing" 

final_test(model_loaded, labels, preprocess, TEST_DIR)

Running validation on subset...


Testing Tomato_Tomato_Yellow_Leaf_Curl_Virus: 100%|██████████| 84/84 [00:02<00:00, 38.94it/s]

{'Potato_Early_blight': {'correct': 49, 'total': 87}, 'Potato_healthy': {'correct': 48, 'total': 63}, 'Potato_Lateblight': {'correct': 51, 'total': 104}, 'Tomato_Bacterial_spot': {'correct': 62, 'total': 100}, 'Tomato_Early_blight': {'correct': 35, 'total': 62}, 'Tomato_healthy': {'correct': 23, 'total': 71}, 'Tomato_Late_blight': {'correct': 45, 'total': 90}, 'Tomato_Leaf_mold': {'correct': 53, 'total': 73}, 'Tomato_Septoria_leaf_spot': {'correct': 46, 'total': 108}, 'Tomato_Tomato_Yellow_Leaf_Curl_Virus': {'correct': 56, 'total': 84}}

Class Name                     | Accuracy  
---------------------------------------------
Potato_Early_blight            |    56.32%
Potato_healthy                 |    76.19%
Potato_Lateblight              |    49.04%
Tomato_Bacterial_spot          |    62.00%
Tomato_Early_blight            |    56.45%
Tomato_healthy                 |    32.39%
Tomato_Late_blight             |    50.00%
Tomato_Leaf_mold               |    72.60%
Tomato_Septoria_leaf_s

In [ ]:
# YOLO v11 + Efficient_Net_B7 (Pipeline)
# Trained on Plant_Doc + Plant_Wild Dataset for 20 epochs with a lr=0.0001 and for another 20 epochs with a lr = 0.00005
# Tested on the dataset provided consisting 10 classes (excluding Tomato_Mosaic Virus, Tomato_Spider_Mites and Tomato_Target_Spot)

In [42]:
model_loaded = torch.load("Efficient_B7_Plant_Doc_Wild.pth",weights_only=False)

In [43]:
# Testing Block

full_dataset = datasets.ImageFolder("Merged_Dataset(Plant_Doc+Plant_Wild)/cropped_training")

labels = full_dataset.classes

preprocess = v2.Compose([
    v2.ToTensor(),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Path to test images
TEST_DIR = "Merged_Dataset(Plant_Doc+Plant_Wild)/cropped_testing" 

final_test(model_loaded, labels, preprocess, TEST_DIR)

Running validation on subset...


Testing Tomato_Tomato_Yellow_Leaf_Curl_Virus: 100%|██████████| 84/84 [00:03<00:00, 24.40it/s]

{'Potato_Early_blight': {'correct': 42, 'total': 87}, 'Potato_healthy': {'correct': 48, 'total': 63}, 'Potato_Lateblight': {'correct': 51, 'total': 104}, 'Tomato_Bacterial_spot': {'correct': 50, 'total': 100}, 'Tomato_Early_blight': {'correct': 29, 'total': 62}, 'Tomato_healthy': {'correct': 28, 'total': 71}, 'Tomato_Late_blight': {'correct': 41, 'total': 90}, 'Tomato_Leaf_mold': {'correct': 51, 'total': 73}, 'Tomato_Septoria_leaf_spot': {'correct': 52, 'total': 108}, 'Tomato_Tomato_Yellow_Leaf_Curl_Virus': {'correct': 53, 'total': 84}}

Class Name                     | Accuracy  
---------------------------------------------
Potato_Early_blight            |    48.28%
Potato_healthy                 |    76.19%
Potato_Lateblight              |    49.04%
Tomato_Bacterial_spot          |    50.00%
Tomato_Early_blight            |    46.77%
Tomato_healthy                 |    39.44%
Tomato_Late_blight             |    45.56%
Tomato_Leaf_mold               |    69.86%
Tomato_Septoria_leaf_s

In [ ]:
# Some other models that were tested:

# 1. YOLO v11 + ViT_L_16 (Trained on Plant_Doc, 20 epochs => lr = 0.0001 + 20 epochs => lr = 0.00005)
#    => Testing Accuracy = 28 % [On 28 classes]

# 2. YOLO v11 + Efficient_Net_B4 (Trained on Plant_Doc for 5 epochs)
#    => Testing Accuracy = 37.1 % [On 13 Classes]

# 3. YOLO v11 + Efficient_Net_B6 (Trained on Plant_Wild+Plant_Doc for 20 epochs)
#    => Testing Accuracy =  54.99 % [On 11 Classes]

In [ ]:
# YOLO v11 + Efficient_Net_B4 (Pipeline)
# Trained on Plant_Doc + Plant_Wild Dataset containing only Potato subclasses
# Tested on the dataset provided containing only subclasses concerning Potato

In [44]:
model_loaded = torch.load("Potato-b4.pth",weights_only=False)

In [45]:
# Testing Block

full_dataset = datasets.ImageFolder("Separated_Merged_Dataset(Plant_Wild+Plant_Doc)/Potato/cropped_training")

labels = full_dataset.classes

preprocess = v2.Compose([
    v2.ToTensor(),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Path to test images
TEST_DIR = "Separated_Merged_Dataset(Plant_Wild+Plant_Doc)/Potato/cropped_testing" 

final_test(model_loaded,labels, preprocess, TEST_DIR)

Running validation on subset...


Testing Potato_Lateblight: 100%|██████████| 104/104 [00:02<00:00, 36.90it/s]

{'Potato_Early_blight': {'correct': 56, 'total': 87}, 'Potato_healthy': {'correct': 59, 'total': 63}, 'Potato_Lateblight': {'correct': 61, 'total': 104}}

Class Name                     | Accuracy  
---------------------------------------------
Potato_Early_blight            |    64.37%
Potato_healthy                 |    93.65%
Potato_Lateblight              |    58.65%
OVERALL ACCURACY ON SUBSET: 69.29%


In [ ]:
# YOLO v11 + ConvexNet-Base (Pipeline)
# Trained on Plant_Doc + Plant_Wild Dataset containing only Potato subclasses
# Tested on the dataset provided containing only subclasses concerning Potato

In [46]:
model_loaded = torch.load("Potato-ConVexNet-Base.pth",weights_only=False)

In [48]:
# Testing Block

full_dataset = datasets.ImageFolder("Separated_Merged_Dataset(Plant_Wild+Plant_Doc)/Potato/cropped_training")

labels = full_dataset.classes

preprocess = v2.Compose([
    v2.ToTensor(),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Path to test images
TEST_DIR = "Separated_Merged_Dataset(Plant_Wild+Plant_Doc)/Potato/cropped_testing" 

final_test(model_loaded,labels,preprocess, TEST_DIR)

c:\Anaconda\envs\env\Lib\site-packages\torchvision\transforms\v2\_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


Running validation on subset...


Testing Potato_Lateblight: 100%|██████████| 104/104 [00:01<00:00, 62.27it/s]

{'Potato_Early_blight': {'correct': 59, 'total': 87}, 'Potato_healthy': {'correct': 56, 'total': 63}, 'Potato_Lateblight': {'correct': 72, 'total': 104}}

Class Name                     | Accuracy  
---------------------------------------------
Potato_Early_blight            |    67.82%
Potato_healthy                 |    88.89%
Potato_Lateblight              |    69.23%
OVERALL ACCURACY ON SUBSET: 73.62%


In [ ]:
# YOLO v11 + Efficient_Net_B7 (Pipeline)
# Trained on Plant_Doc + Plant_Wild Dataset containing only Tomato subclasses
# Tested on the dataset provided containing only subclasses concerning Tomato(excluding Spider mites, Target Spot and Mosaic Virus)

In [49]:
model_loaded = torch.load("Tomato-b7.pth",weights_only=False)

In [52]:
# Testing Block

full_dataset = datasets.ImageFolder("Separated_Merged_Dataset(Plant_Wild+Plant_Doc)/Tomato/cropped_training")

labels = full_dataset.classes

preprocess = v2.Compose([
    v2.ToTensor(),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Path to test images
TEST_DIR = "Separated_Merged_Dataset(Plant_Wild+Plant_Doc)/Tomato/cropped_testing" 

final_test(model_loaded,labels,preprocess, TEST_DIR)

c:\Anaconda\envs\env\Lib\site-packages\torchvision\transforms\v2\_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


Running validation on subset...


Testing Tomato_Tomato_Yellow_Leaf_Curl_Virus: 100%|██████████| 84/84 [00:03<00:00, 23.80it/s]

{'Tomato_Bacterial_spot': {'correct': 49, 'total': 100}, 'Tomato_Early_blight': {'correct': 32, 'total': 62}, 'Tomato_healthy': {'correct': 38, 'total': 71}, 'Tomato_Late_blight': {'correct': 55, 'total': 90}, 'Tomato_Leaf_mold': {'correct': 55, 'total': 73}, 'Tomato_Septoria_leaf_spot': {'correct': 64, 'total': 108}, 'Tomato_Tomato_Yellow_Leaf_Curl_Virus': {'correct': 59, 'total': 84}}

Class Name                     | Accuracy  
---------------------------------------------
Tomato_Bacterial_spot          |    49.00%
Tomato_Early_blight            |    51.61%
Tomato_healthy                 |    53.52%
Tomato_Late_blight             |    61.11%
Tomato_Leaf_mold               |    75.34%
Tomato_Septoria_leaf_spot      |    59.26%
Tomato_Tomato_Yellow_Leaf_Curl_Virus |    70.24%
OVERALL ACCURACY ON SUBSET: 59.86%


In [53]:
# YOLO v11 + ConvexNet-Base (Pipeline)
# Trained on Plant_Doc + Plant_Wild Dataset containing only Tomato subclasses
# Tested on the dataset provided containing only subclasses concerning Tomato(excluding Spider mites, Target Spot and Mosaic Virus)

In [54]:
model_loaded = torch.load("Tomato-ConVexNet-Base.pth",weights_only=False)

In [56]:
# Testing Block

full_dataset = datasets.ImageFolder("Separated_Merged_Dataset(Plant_Wild+Plant_Doc)/Tomato/cropped_training")

labels = full_dataset.classes

preprocess = v2.Compose([
    v2.ToTensor(),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Path to test images
TEST_DIR = "Separated_Merged_Dataset(Plant_Wild+Plant_Doc)/Tomato/cropped_testing" 

final_test(model_loaded,labels,preprocess, TEST_DIR)

c:\Anaconda\envs\env\Lib\site-packages\torchvision\transforms\v2\_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


Running validation on subset...


Testing Tomato_Tomato_Yellow_Leaf_Curl_Virus: 100%|██████████| 84/84 [00:01<00:00, 65.53it/s]

{'Tomato_Bacterial_spot': {'correct': 51, 'total': 100}, 'Tomato_Early_blight': {'correct': 40, 'total': 62}, 'Tomato_healthy': {'correct': 38, 'total': 71}, 'Tomato_Late_blight': {'correct': 61, 'total': 90}, 'Tomato_Leaf_mold': {'correct': 60, 'total': 73}, 'Tomato_Septoria_leaf_spot': {'correct': 67, 'total': 108}, 'Tomato_Tomato_Yellow_Leaf_Curl_Virus': {'correct': 74, 'total': 84}}

Class Name                     | Accuracy  
---------------------------------------------
Tomato_Bacterial_spot          |    51.00%
Tomato_Early_blight            |    64.52%
Tomato_healthy                 |    53.52%
Tomato_Late_blight             |    67.78%
Tomato_Leaf_mold               |    82.19%
Tomato_Septoria_leaf_spot      |    62.04%
Tomato_Tomato_Yellow_Leaf_Curl_Virus |    88.10%
OVERALL ACCURACY ON SUBSET: 66.50%


In [ ]:
# YOLO v11 + ConvexNet-Large (Pipeline)
# Trained on Plant_Doc + Plant_Wild Dataset for 10 epochs
# Tested on the dataset provided consisting 10 classes (excluding Tomato_Mosaic Virus, Tomato_Spider_Mites and Tomato_Target_Spot)

In [57]:
model_loaded = torch.load("ConVexNet-Large-Merged-Plant_Doc_Wild.pth",weights_only=False)

In [59]:
# Testing Block

full_dataset = datasets.ImageFolder("Merged_Dataset(Plant_Doc+Plant_Wild)/cropped_training")

labels = full_dataset.classes

preprocess = v2.Compose([
    v2.ToTensor(),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Path to test images
TEST_DIR = "Merged_Dataset(Plant_Doc+Plant_Wild)/cropped_testing" 

final_test(model_loaded,labels,preprocess, TEST_DIR)

c:\Anaconda\envs\env\Lib\site-packages\torchvision\transforms\v2\_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


Running validation on subset...


Testing Tomato_Tomato_Yellow_Leaf_Curl_Virus: 100%|██████████| 84/84 [00:01<00:00, 63.52it/s]

{'Potato_Early_blight': {'correct': 37, 'total': 87}, 'Potato_healthy': {'correct': 37, 'total': 63}, 'Potato_Lateblight': {'correct': 59, 'total': 104}, 'Tomato_Bacterial_spot': {'correct': 53, 'total': 100}, 'Tomato_Early_blight': {'correct': 45, 'total': 62}, 'Tomato_healthy': {'correct': 35, 'total': 71}, 'Tomato_Late_blight': {'correct': 45, 'total': 90}, 'Tomato_Leaf_mold': {'correct': 56, 'total': 73}, 'Tomato_Septoria_leaf_spot': {'correct': 51, 'total': 108}, 'Tomato_Tomato_Yellow_Leaf_Curl_Virus': {'correct': 60, 'total': 84}}

Class Name                     | Accuracy  
---------------------------------------------
Potato_Early_blight            |    42.53%
Potato_healthy                 |    58.73%
Potato_Lateblight              |    56.73%
Tomato_Bacterial_spot          |    53.00%
Tomato_Early_blight            |    72.58%
Tomato_healthy                 |    49.30%
Tomato_Late_blight             |    50.00%
Tomato_Leaf_mold               |    76.71%
Tomato_Septoria_leaf_s

In [ ]:
# --------------------------------------------------------------------------[Best Model as of Now]--------------------------------------------------------------------------------------
# YOLO v11 + ConvexNet-Base (Pipeline) 
# Trained on Plant_Doc + Plant_Wild Dataset for 20 epochs, and included a new transformation during training transforms.equalize()
# Tested on the dataset provided consisting 10 classes (excluding Tomato_Mosaic Virus, Tomato_Spider_Mites and Tomato_Target_Spot)

In [60]:
model_loaded = torch.load("ConVexNet-Base-Merged-Plant_Doc_Wild.pth",weights_only=False)

In [62]:
# Testing Block

full_dataset = datasets.ImageFolder("Merged_Dataset(Plant_Doc+Plant_Wild)/cropped_training")

labels = full_dataset.classes

preprocess = v2.Compose([
    v2.ToTensor(),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Path to test images
TEST_DIR = "Merged_Dataset(Plant_Doc+Plant_Wild)/cropped_testing" 

final_test(model_loaded,labels,preprocess, TEST_DIR)

c:\Anaconda\envs\env\Lib\site-packages\torchvision\transforms\v2\_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


Running validation on subset...


Testing Tomato_Tomato_Yellow_Leaf_Curl_Virus: 100%|██████████| 84/84 [00:01<00:00, 62.71it/s]

{'Potato_Early_blight': {'correct': 47, 'total': 87}, 'Potato_healthy': {'correct': 50, 'total': 63}, 'Potato_Lateblight': {'correct': 47, 'total': 104}, 'Tomato_Bacterial_spot': {'correct': 65, 'total': 100}, 'Tomato_Early_blight': {'correct': 31, 'total': 62}, 'Tomato_healthy': {'correct': 49, 'total': 71}, 'Tomato_Late_blight': {'correct': 54, 'total': 90}, 'Tomato_Leaf_mold': {'correct': 56, 'total': 73}, 'Tomato_Septoria_leaf_spot': {'correct': 50, 'total': 108}, 'Tomato_Tomato_Yellow_Leaf_Curl_Virus': {'correct': 58, 'total': 84}}

Class Name                     | Accuracy  
---------------------------------------------
Potato_Early_blight            |    54.02%
Potato_healthy                 |    79.37%
Potato_Lateblight              |    45.19%
Tomato_Bacterial_spot          |    65.00%
Tomato_Early_blight            |    50.00%
Tomato_healthy                 |    69.01%
Tomato_Late_blight             |    60.00%
Tomato_Leaf_mold               |    76.71%
Tomato_Septoria_leaf_s

In [ ]:
# YOLO v11 + ConvexNet-Base (Pipeline)
# Trained on Plant_Doc + Plant_Wild Dataset for 20 epochs, and included a new transformation during training transforms.equalize()
# Tried resizing the images from 224x224 to 299x299
# Tested on the dataset provided consisting 10 classes (excluding Tomato_Mosaic Virus, Tomato_Spider_Mites and Tomato_Target_Spot)

In [63]:
model_loaded = torch.load("ConVexNet-Base-Merged-Plant_Doc_Wild_299.pth",weights_only=False)

In [65]:
# Testing Block

full_dataset = datasets.ImageFolder("Merged_Dataset(Plant_Doc+Plant_Wild)/cropped_training_299")

labels = full_dataset.classes

preprocess = v2.Compose([
    v2.ToTensor(),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Path to test images
TEST_DIR = "Merged_Dataset(Plant_Doc+Plant_Wild)/cropped_testing_299" 

final_test(model_loaded,labels,preprocess, TEST_DIR)

c:\Anaconda\envs\env\Lib\site-packages\torchvision\transforms\v2\_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


Running validation on subset...


Testing Tomato_Tomato_Yellow_Leaf_Curl_Virus: 100%|██████████| 84/84 [00:01<00:00, 60.87it/s]

{'Potato_Early_blight': {'correct': 34, 'total': 87}, 'Potato_healthy': {'correct': 43, 'total': 63}, 'Potato_Lateblight': {'correct': 55, 'total': 104}, 'Tomato_Bacterial_spot': {'correct': 77, 'total': 100}, 'Tomato_Early_blight': {'correct': 30, 'total': 62}, 'Tomato_healthy': {'correct': 43, 'total': 71}, 'Tomato_Late_blight': {'correct': 56, 'total': 90}, 'Tomato_Leaf_mold': {'correct': 57, 'total': 73}, 'Tomato_Septoria_leaf_spot': {'correct': 44, 'total': 108}, 'Tomato_Tomato_Yellow_Leaf_Curl_Virus': {'correct': 65, 'total': 84}}

Class Name                     | Accuracy  
---------------------------------------------
Potato_Early_blight            |    39.08%
Potato_healthy                 |    68.25%
Potato_Lateblight              |    52.88%
Tomato_Bacterial_spot          |    77.00%
Tomato_Early_blight            |    48.39%
Tomato_healthy                 |    60.56%
Tomato_Late_blight             |    62.22%
Tomato_Leaf_mold               |    78.08%
Tomato_Septoria_leaf_s

In [ ]:
# YOLO v11 + ConvexNet-Small (Pipeline)
# Used the small model to make sure ConvexNet-Base was not overfitting the training set
# Trained on Plant_Doc + Plant_Wild Dataset for 20 epochs, and included a new transformation during training transforms.equalize()
# Tested on the dataset provided consisting 10 classes (excluding Tomato_Mosaic Virus, Tomato_Spider_Mites and Tomato_Target_Spot)

In [67]:
model_loaded = torch.load("ConVexNet-Small-Merged-Plant_Doc_Wild.pth",weights_only=False)

In [68]:
# Testing Block

full_dataset = datasets.ImageFolder("Merged_Dataset(Plant_Doc+Plant_Wild)/cropped_training")

labels = full_dataset.classes

preprocess = v2.Compose([
    v2.ToTensor(),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Path to test images
TEST_DIR = "Merged_Dataset(Plant_Doc+Plant_Wild)/cropped_testing" 

final_test(model_loaded,labels,preprocess, TEST_DIR)

c:\Anaconda\envs\env\Lib\site-packages\torchvision\transforms\v2\_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


Running validation on subset...


Testing Tomato_Tomato_Yellow_Leaf_Curl_Virus: 100%|██████████| 84/84 [00:05<00:00, 15.73it/s]

{'Potato_Early_blight': {'correct': 48, 'total': 87}, 'Potato_healthy': {'correct': 41, 'total': 63}, 'Potato_Lateblight': {'correct': 47, 'total': 104}, 'Tomato_Bacterial_spot': {'correct': 54, 'total': 100}, 'Tomato_Early_blight': {'correct': 36, 'total': 62}, 'Tomato_healthy': {'correct': 36, 'total': 71}, 'Tomato_Late_blight': {'correct': 61, 'total': 90}, 'Tomato_Leaf_mold': {'correct': 48, 'total': 73}, 'Tomato_Septoria_leaf_spot': {'correct': 50, 'total': 108}, 'Tomato_Tomato_Yellow_Leaf_Curl_Virus': {'correct': 45, 'total': 84}}

Class Name                     | Accuracy  
---------------------------------------------
Potato_Early_blight            |    55.17%
Potato_healthy                 |    65.08%
Potato_Lateblight              |    45.19%
Tomato_Bacterial_spot          |    54.00%
Tomato_Early_blight            |    58.06%
Tomato_healthy                 |    50.70%
Tomato_Late_blight             |    67.78%
Tomato_Leaf_mold               |    65.75%
Tomato_Septoria_leaf_s